# Aim 1 models: Insurance and Post-ICU Transitions (MIMIC-IV v3.1)

**Data:** `ctheg-scores.ins_icu.aim1_primary_18_64` (primary) and `aim1_older_65plus` (separate analysis),
built by `01_aim1_cohort.sql` + `02_aim1_covariates.sql`.

**Privacy rules for this notebook**
- Row-level MIMIC data stays inside this Colab session. Do not download it, print rows, or save it to Drive/GitHub.
- Do not use Colab's built-in AI features (Gemini code assist / "Explain error" / data-science agent) on this notebook.
- Before saving or sharing the notebook: **Edit → Clear all outputs**. Share only the results file written at the end
  (`aim1_results.xlsx`), which contains model estimates and aggregate counts only.

**Pre-specified primary model (fixed 2026-11-12, before any adjusted outcome model was run)**
- Exposure: admission-recorded Medicaid vs Private (reference). Not verified coverage; no dual-eligibility inference.
- Covariates: age (+ age²), sex, race group, first-day SOFA, Charlson (age points removed), admission type,
  invasive ventilation, vasopressors, log(1 + ICU LOS), **ICU type** (added 2026-11-12 after Table 1 showed CVICU/MICU
  imbalance; added before outcome modelling), era (anchor_year_group).
- Outcome 1: discharge destination (home, home health, SNF, rehab, LTACH, other), multinomial logit;
  reported as average adjusted predicted probabilities (g-computation) with bootstrap 95% CIs.
- Outcome 2: days from final ICU exit to hospital discharge, two-part model
  (logit for any post-ICU ward stay > 4 h; gamma GLM with log link for days among those with a stay).
- Two-sided tests. Associations, not causal effects. Single centre (BIDMC).
- Robustness: IPW (stabilised, trimmed). Sensitivity: excl. AMA/acute transfers; 1-h timing rule; + language.
- Exploratory: Oaxaca–Blinder decomposition; post-ICU days stratified by destination group; 65+ analysis.
- Technical note (2026-11-12, after a B=50 test run): ICU units with ~60 patients (PACU, Medicine, etc.) merged into
  'Mixed' ICU type because a separate tiny group caused non-convergence in bootstrap resamples; non-converged
  bootstrap reps are dropped and counted.

In [ ]:
# ---- Settings ---------------------------------------------------------------
PROJECT  = "ctheg-scores"
SEED     = 20261112
B        = 500     # FINAL run: bootstrap reps for primary estimates
B_SENS   = 200     # FINAL run: reps for sensitivity / 65+ / Oaxaca / by-destination
OUTFILE  = "aim1_results.xlsx"
MIN_CELL = 11      # aggregate counts below this are suppressed in exported tables

import time, warnings
import numpy as np
import pandas as pd
import statsmodels
import statsmodels.api as sm
import statsmodels.formula.api as smf
import patsy

from statsmodels.tools.sm_exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)   # convergence is checked explicitly below

class NotConverged(Exception):
    pass

def check(res, label):
    ok = res.mle_retvals.get("converged", True) if hasattr(res, "mle_retvals") and res.mle_retvals else getattr(res, "converged", True)
    if not ok:
        raise NotConverged(label)
    return res
pd.set_option("display.width", 200, "display.max_columns", 50)
print("statsmodels", statsmodels.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)

In [ ]:
# ---- Load the analysis views from BigQuery (no file downloads) ---------------
from google.colab import auth
auth.authenticate_user()          # sign in with the Google account linked to PhysioNet
from google.cloud import bigquery
client = bigquery.Client(project=PROJECT)

COLS = """ins_cat, dc_group, post_icu_days, any_post_icu_stay, dc_ama, dc_acute_transfer,
          time_valid_strict, age_at_admit, gender, race_cat, sofa_day1, cci_no_age,
          adm_type_cat, invasive_vent, vasopressor, icu_los_days_total, first_careunit,
          anchor_year_group, language_cat"""

raw18 = client.query(f"SELECT {COLS} FROM `{PROJECT}.ins_icu.aim1_primary_18_64`").to_dataframe()
raw65 = client.query(f"SELECT {COLS} FROM `{PROJECT}.ins_icu.aim1_older_65plus`").to_dataframe()
print("18-64:", raw18.shape, "| 65+:", raw65.shape)

In [ ]:
# ---- Variable preparation -----------------------------------------------------
DC_LEVELS = ["home", "home_health", "snf", "rehab", "ltach", "other"]   # code 0 = home (reference)

ICU_MAP = {
    "Cardiac Vascular Intensive Care Unit (CVICU)":       "Cardiac",
    "Coronary Care Unit (CCU)":                           "Cardiac",
    "Medical Intensive Care Unit (MICU)":                 "Medical",
    "Medical/Surgical Intensive Care Unit (MICU/SICU)":   "Mixed",
    "Surgical Intensive Care Unit (SICU)":                "Surgical/Trauma",
    "Trauma SICU (TSICU)":                                "Surgical/Trauma",
    "Neuro Surgical Intensive Care Unit (Neuro SICU)":    "Neuro",
    "Neuro Intermediate":                                 "Neuro",
    "Neuro Stepdown":                                     "Neuro",
}   # anything else (PACU, Medicine, 'Intensive Care Unit (ICU)', small surgical units; ~60 patients) -> 'Mixed'
#     (merged 2026-11-12: a separate tiny group caused non-convergence in bootstrap resamples)

REF = {"race_cat": "White", "adm_type_cat": "Emergency", "icu_type": "Medical",
       "anchor_year_group": "2008 - 2010", "language_cat": "English"}

def prep(raw, exposure_levels):
    d = raw[raw["ins_cat"].isin(exposure_levels)].copy()
    d["icu_type"]    = d["first_careunit"].map(ICU_MAP).fillna("Mixed")
    d["female"]      = (d["gender"] == "F").astype(int)
    d["age_c"]       = (d["age_at_admit"] - 50) / 10            # per 10 years, centred at 50
    d["log_icu_los"] = np.log1p(d["icu_los_days_total"].astype(float))
    d["vent"]        = d["invasive_vent"].astype(int)
    d["pressor"]     = d["vasopressor"].astype(int)
    d["any_stay"]    = d["any_post_icu_stay"].astype(int)
    d["days_total"]  = np.where(d["any_stay"] == 1, d["post_icu_days"].astype(float), 0.0)  # two-part: <=4 h counted as 0
    d["post_icu_days"] = d["post_icu_days"].astype(float)
    d["sofa_day1"]   = d["sofa_day1"].astype(float)
    d["cci_no_age"]  = d["cci_no_age"].astype(float)
    d["facility"]    = d["dc_group"].isin(["snf", "rehab", "ltach"]).astype(int)
    for c in ["dc_ama", "dc_acute_transfer", "time_valid_strict"]:
        d[c] = d[c].astype(bool)
    d["dc_code"] = pd.Categorical(d["dc_group"], categories=DC_LEVELS).codes
    d["ins"] = pd.Categorical(d["ins_cat"], categories=exposure_levels)   # first level = reference
    for c, ref in REF.items():                                            # fixed level order, reference first
        levels = [ref] + sorted(x for x in d[c].dropna().unique() if x != ref)
        d[c] = pd.Categorical(d[c], categories=levels)
    n_missing = d[["sofa_day1", "cci_no_age", "dc_code"]].isna().sum()
    assert (d["dc_code"] >= 0).all(), "unexpected discharge group"
    print(f"n = {len(d):,} | missing:", n_missing.to_dict())
    return d.dropna(subset=["sofa_day1", "cci_no_age"]).reset_index(drop=True)

COV = ("age_c + I(age_c**2) + female + C(race_cat) + sofa_day1 + cci_no_age + C(adm_type_cat)"
       " + vent + pressor + log_icu_los + C(icu_type) + C(anchor_year_group)")
COV_LANG = COV + " + C(language_cat)"
EXPO = "C(ins)"

d18 = prep(raw18, ["Private", "Medicaid"])
d18["medicaid"] = (d18["ins_cat"] == "Medicaid").astype(int)
print(d18["ins"].value_counts())

In [ ]:
# ---- Helpers: counterfactual prediction, bootstrap, summaries ------------------
def set_exposure(d, level):
    dd = d.copy()
    dd["ins"] = pd.Categorical([level] * len(dd), categories=d["ins"].cat.categories)
    return dd

def bootstrap(fn, d, reps, seed=SEED, label=""):
    """Nonparametric bootstrap over patients (one row per patient). fn(d) -> dict of named estimates."""
    rng, out, nc, err, t0 = np.random.default_rng(seed), [], 0, 0, time.time()
    for b in range(reps):
        db = d.iloc[rng.integers(0, len(d), len(d))].reset_index(drop=True)
        try:
            out.append(fn(db))
        except NotConverged:
            nc += 1          # dropped; count reported
        except Exception:
            err += 1
    print(f"[{label}] bootstrap {len(out)}/{reps} used | non-converged dropped: {nc} | errors: {err} | {time.time() - t0:.0f}s")
    return pd.DataFrame(out)

def summarise(point, boot):
    rows = []
    for k, v in point.items():
        lo, hi = np.nanpercentile(boot[k], [2.5, 97.5]) if k in boot else (np.nan, np.nan)
        rows.append({"estimate": k, "value": v, "ci_low": lo, "ci_high": hi})
    return pd.DataFrame(rows)

def coef_table(res, exp=True, name=""):
    ci = res.conf_int()
    t = pd.DataFrame({"coef": res.params, "se": res.bse, "p": res.pvalues,
                      "ci_low": ci.iloc[:, 0], "ci_high": ci.iloc[:, 1]})
    if exp:
        t["ratio"], t["ratio_ci_low"], t["ratio_ci_high"] = np.exp(t["coef"]), np.exp(t["ci_low"]), np.exp(t["ci_high"])
    t.insert(0, "model", name)
    return t.reset_index().rename(columns={"index": "term"})

In [ ]:
# ---- Outcome 1: discharge destination (multinomial logit + g-computation) ------
def fit_mnlogit(d, cov=COV, start=None):
    res = smf.mnlogit(f"dc_code ~ {EXPO} + {cov}", d).fit(
        disp=0, maxiter=60, method="newton", start_params=start)
    return check(res, "mnlogit")

def dest_estimates(d, cov=COV, start=None):
    res = fit_mnlogit(d, cov, start)
    levels, ref = list(d["ins"].cat.categories), d["ins"].cat.categories[0]
    probs = {lvl: np.asarray(res.predict(set_exposure(d, lvl))).mean(axis=0) for lvl in levels}
    out = {}
    for lvl in levels:
        for j, dest in enumerate(DC_LEVELS):
            out[f"P({dest}) | {lvl}"] = probs[lvl][j]
    for lvl in levels[1:]:
        for j, dest in enumerate(DC_LEVELS):
            out[f"Diff P({dest}) | {lvl} - {ref}"] = probs[lvl][j] - probs[ref][j]
    return out, res

t0 = time.time()
dest_point, dest_fit = dest_estimates(d18)
print(f"multinomial fit: {time.time() - t0:.1f}s, converged={dest_fit.mle_retvals['converged']}")
_start = dest_fit.params.values.flatten(order="F")
dest_boot = bootstrap(lambda x: dest_estimates(x, start=_start)[0], d18, B, label="destination")
dest_tab = summarise(dest_point, dest_boot)

# RRR table (supplement): rows = terms, one block per destination vs home
mn = dest_fit
rrr = []
for j, dest in enumerate(DC_LEVELS[1:]):
    p, se = mn.params.iloc[:, j], mn.bse.iloc[:, j]
    rrr.append(pd.DataFrame({"outcome_vs_home": dest, "term": p.index, "rrr": np.exp(p),
                             "ci_low": np.exp(p - 1.96 * se), "ci_high": np.exp(p + 1.96 * se),
                             "p": mn.pvalues.iloc[:, j]}))
rrr_tab = pd.concat(rrr, ignore_index=True)
dest_tab.round(4)

In [ ]:
# ---- Outcome 2: post-ICU hospital days (two-part model) ------------------------
def days_estimates(d, cov=COV):
    p1 = check(smf.logit(f"any_stay ~ {EXPO} + {cov}", d).fit(disp=0, maxiter=60), "logit part 1")
    pos = d[d["any_stay"] == 1]
    p2 = check(smf.glm(f"post_icu_days ~ {EXPO} + {cov}", pos,
                 family=sm.families.Gamma(sm.families.links.Log())).fit(), "gamma part 2")
    levels, ref = list(d["ins"].cat.categories), d["ins"].cat.categories[0]
    out, e = {}, {}
    for lvl in levels:
        dd = set_exposure(d, lvl)
        pa, cd = np.asarray(p1.predict(dd)), np.asarray(p2.predict(dd))
        e[lvl] = (pa.mean(), cd.mean(), (pa * cd).mean())
        out[f"P(any post-ICU stay) | {lvl}"]   = e[lvl][0]
        out[f"E(days | stay) | {lvl}"]         = e[lvl][1]
        out[f"E(days) overall | {lvl}"]        = e[lvl][2]
    for lvl in levels[1:]:
        out[f"Diff P(any stay) | {lvl} - {ref}"]   = e[lvl][0] - e[ref][0]
        out[f"Diff E(days | stay) | {lvl} - {ref}"] = e[lvl][1] - e[ref][1]
        out[f"Diff E(days) overall | {lvl} - {ref}"] = e[lvl][2] - e[ref][2]
        out[f"Ratio E(days | stay) | {lvl} / {ref}"] = float(np.exp(p2.params[f"C(ins)[T.{lvl}]"]))
    return out, (p1, p2)

days_point, (days_p1, days_p2) = days_estimates(d18)
days_boot = bootstrap(lambda x: days_estimates(x)[0], d18, B, label="days")
days_tab = summarise(days_point, days_boot)
days_coef = pd.concat([coef_table(days_p1, name="part1_logit_any_stay (OR)"),
                       coef_table(days_p2, name="part2_gamma_log_days (ratio)")], ignore_index=True)
days_tab.round(4)

In [ ]:
# ---- Robustness: inverse probability weighting (Medicaid vs Private) ----------
def ipw_weights(d, cov=COV, trim=(0.01, 0.99)):
    ps = np.asarray(check(smf.logit(f"medicaid ~ {cov}", d).fit(disp=0, maxiter=60), "PS").predict(d))
    t, pt = d["medicaid"].values, d["medicaid"].mean()
    w = np.where(t == 1, pt / ps, (1 - pt) / (1 - ps))           # stabilised ATE weights
    lo, hi = np.quantile(w, trim)
    return np.clip(w, lo, hi), ps

def ipw_estimates(d, cov=COV):
    w, _ = ipw_weights(d, cov)
    t = d["medicaid"].values
    def wmean(x, g): return np.average(x[t == g], weights=w[t == g])
    out = {}
    for dest in DC_LEVELS:
        x = (d["dc_group"] == dest).values.astype(float)
        out[f"IPW Diff P({dest}) | Medicaid - Private"] = wmean(x, 1) - wmean(x, 0)
    out["IPW Diff P(any stay) | Medicaid - Private"]   = wmean(d["any_stay"].values, 1) - wmean(d["any_stay"].values, 0)
    out["IPW Diff E(days) overall | Medicaid - Private"] = wmean(d["days_total"].values, 1) - wmean(d["days_total"].values, 0)
    return out

def balance_table(d, cov=COV):
    X = patsy.dmatrix(cov, d, return_type="dataframe").drop(columns="Intercept")
    w, ps = ipw_weights(d, cov)
    t = d["medicaid"].values == 1
    def smd(wt):
        m1, m0 = np.average(X[t], axis=0, weights=wt[t]), np.average(X[~t], axis=0, weights=wt[~t])
        v1, v0 = X[t].var().values, X[~t].var().values
        return (m1 - m0) / np.sqrt((v1 + v0) / 2)
    bt = pd.DataFrame({"term": X.columns, "smd_unweighted": smd(np.ones(len(d))), "smd_ipw": smd(w)})
    print(f"max |SMD| unweighted {bt.smd_unweighted.abs().max():.3f} -> IPW {bt.smd_ipw.abs().max():.3f}; "
          f"weights range {w.min():.2f}-{w.max():.2f}; PS range {ps.min():.3f}-{ps.max():.3f}")
    return bt

bal_tab = balance_table(d18)
ipw_point = ipw_estimates(d18)
ipw_boot = bootstrap(ipw_estimates, d18, B, label="IPW")
ipw_tab = summarise(ipw_point, ipw_boot)
ipw_tab.round(4)

In [ ]:
# ---- Sensitivity analyses (point estimates + bootstrap CIs) -------------------
def key_diffs(d, cov=COV):
    a, _ = dest_estimates(d, cov)
    b, _ = days_estimates(d, cov)
    keep = {k: v for k, v in {**a, **b}.items() if k.startswith("Diff")}
    return keep

SENS = {
    "S1 excl. AMA and acute-hospital transfers": (d18[~(d18.dc_ama | d18.dc_acute_transfer)].reset_index(drop=True), COV),
    "S2 1-hour timing tolerance":                (d18[d18.time_valid_strict].reset_index(drop=True), COV),
    "S3 + language (English / non-English)":     (d18, COV_LANG),
}
sens_rows = []
for name, (dsub, cov) in SENS.items():
    pt = key_diffs(dsub, cov)
    bt = bootstrap(lambda x, c=cov: key_diffs(x, c), dsub, B_SENS, label=name)
    s = summarise(pt, bt); s.insert(0, "analysis", name); s.insert(1, "n", len(dsub))
    sens_rows.append(s)
sens_tab = pd.concat(sens_rows, ignore_index=True)
sens_tab.round(4)

In [ ]:
# ---- Exploratory: Oaxaca–Blinder (pooled, Jann/Fortin with group indicator) ---
# Linear models; outcomes = discharge to a facility (SNF/rehab/LTACH, LPM) and overall post-ICU days.
# Gap = mean(Medicaid) - mean(Private). Explained = (Xbar_M - Xbar_P)' beta_pooled, by covariate block.
BLOCKS = {"age": ["age_c", "I(age_c ** 2)"], "sex": ["female"], "race": ["C(race_cat)"],
          "severity (SOFA, Charlson)": ["sofa_day1", "cci_no_age"], "admission type": ["C(adm_type_cat)"],
          "organ support (vent, pressor)": ["vent", "pressor"], "ICU LOS": ["log_icu_los"],
          "ICU type": ["C(icu_type)"], "era": ["C(anchor_year_group)"]}

def oaxaca(d, y):
    X = patsy.dmatrix(COV, d, return_type="dataframe")
    t = d["medicaid"].values == 1
    beta = sm.OLS(d[y].values, pd.concat([X, d[["medicaid"]]], axis=1)).fit().params.drop("medicaid")
    dx = X[t].mean() - X[~t].mean()
    contrib = dx * beta
    gap = d[y].values[t].mean() - d[y].values[~t].mean()
    out = {f"OB {y}: gap": gap, f"OB {y}: explained": contrib.sum(),
           f"OB {y}: unexplained": gap - contrib.sum()}
    for blk, pats in BLOCKS.items():
        cols = [c for c in X.columns if any(c == p or c.startswith(p + "[") for p in pats)]
        out[f"OB {y}: explained by {blk}"] = contrib[cols].sum()
    return out

ob_fn = lambda x: {**oaxaca(x, "facility"), **oaxaca(x, "days_total")}
ob_tab = summarise(ob_fn(d18), bootstrap(ob_fn, d18, B_SENS, label="Oaxaca"))
ob_tab.round(4)

In [ ]:
# ---- Exploratory: post-ICU days by discharge destination group ---------------
# Destination is plausibly ON the pathway insurance -> days (placement takes time), so this is
# stratification for description, NOT an adjustment to the primary model.
STRATA = {"home / home health": ["home", "home_health"], "facility (SNF / rehab / LTACH)": ["snf", "rehab", "ltach"]}

def days_by_dest(d, cov=COV):
    out = {}
    for name, grp in STRATA.items():
        ds = d[d["dc_group"].isin(grp) & (d["any_stay"] == 1)]
        g = check(smf.glm(f"post_icu_days ~ {EXPO} + {cov}", ds,
                          family=sm.families.Gamma(sm.families.links.Log())).fit(), "gamma strat")
        e = {lvl: np.asarray(g.predict(set_exposure(ds, lvl))).mean() for lvl in ["Private", "Medicaid"]}
        out[f"E(days | stay) | Private | {name}"]  = e["Private"]
        out[f"E(days | stay) | Medicaid | {name}"] = e["Medicaid"]
        out[f"Diff E(days | stay) | Medicaid - Private | {name}"] = e["Medicaid"] - e["Private"]
        out[f"Ratio E(days | stay) | Medicaid / Private | {name}"] = float(np.exp(g.params["C(ins)[T.Medicaid]"]))
    return out

strat_tab = summarise(days_by_dest(d18), bootstrap(days_by_dest, d18, B_SENS, label="days by destination"))
strat_tab.round(4)

In [ ]:
# ---- Separate analysis: adults 65+ (reference = Medicare; admission-recorded categories) ----
LEV65 = [l for l in ["Medicare", "Medicaid", "Private", "Other"]
         if (raw65["ins_cat"] == l).sum() >= 200]          # drop very small categories
d65 = prep(raw65, LEV65)
print(d65["ins"].value_counts())
d65_point, _ = dest_estimates(d65)
d65_days, _  = days_estimates(d65)
p65 = {**d65_point, **d65_days}
b65 = bootstrap(lambda x: {**dest_estimates(x)[0], **days_estimates(x)[0]}, d65, B_SENS, label="65+")
older_tab = summarise(p65, b65)
older_tab.round(4)

In [ ]:
# ---- Aggregate descriptive tables with small-cell suppression -----------------
def crude_dest(d, label):
    ct = pd.crosstab(d["dc_group"], d["ins_cat"])
    pct = (ct / ct.sum() * 100).round(1)
    out = (ct.astype(str) + " (" + pct.astype(str) + "%)").where(ct >= MIN_CELL, f"<{MIN_CELL}")
    out.insert(0, "cohort", label)
    return out.reset_index()

crude_tab = pd.concat([crude_dest(d18, "18-64"), crude_dest(d65, "65+")], ignore_index=True)
crude_tab

In [ ]:
# ---- Export results only (no row-level data) ------------------------------------
meta = pd.DataFrame({"item": ["run_date", "statsmodels", "pandas", "numpy", "seed", "B", "B_SENS",
                              "n_18_64", "n_65plus", "min_cell"],
                     "value": [pd.Timestamp.now().isoformat(), statsmodels.__version__, pd.__version__,
                               np.__version__, SEED, B, B_SENS, len(d18), len(d65), MIN_CELL]})
with pd.ExcelWriter(OUTFILE) as xw:
    meta.to_excel(xw, sheet_name="run_info", index=False)
    crude_tab.to_excel(xw, sheet_name="crude_destination", index=False)
    dest_tab.to_excel(xw, sheet_name="dest_adjusted_probs", index=False)
    rrr_tab.to_excel(xw, sheet_name="dest_mnlogit_rrr", index=False)
    days_tab.to_excel(xw, sheet_name="days_two_part", index=False)
    days_coef.to_excel(xw, sheet_name="days_coefficients", index=False)
    ipw_tab.to_excel(xw, sheet_name="ipw", index=False)
    bal_tab.to_excel(xw, sheet_name="ipw_balance", index=False)
    sens_tab.to_excel(xw, sheet_name="sensitivity", index=False)
    ob_tab.to_excel(xw, sheet_name="oaxaca_exploratory", index=False)
    strat_tab.to_excel(xw, sheet_name="days_by_destination_expl", index=False)
    older_tab.to_excel(xw, sheet_name="age65plus", index=False)
print("wrote", OUTFILE)

try:
    from google.colab import files
    files.download(OUTFILE)      # results file only
except ImportError:
    pass